# Informe Matemático — Trabajo 2

Este notebook contiene la especificación teórica, algoritmos y resultados de validación para el modelo M/M/k/k (Erlang B).


## Modelo M/M/k/k (Erlang B) — Diagrama y derivación

El modelo M/M/k/k (con fuente infinita) consiste en:
- k servidores paralelos (cada servidor atiende a un cliente con tiempo de servicio exponencial de parámetro $\mu$),
- llegadas Poisson con tasa $\lambda$,
- sin espacio de espera: si llegan con los $k$ servidores ocupados la llegada se bloquea y se pierde (no hay cola).

Las tasas de transición (birth–death) son:
- tasa de nacimiento (llegada) $\lambda_n = \lambda$ para $n=0,\dots,k-1$; $\lambda_k = 0$ (llegadas bloqueadas en $k$).
- tasa de muerte (salida) $\mu_n = n\mu$ para $n \ge 1$ (hay $n$ servidores ocupados que completan servicio con tasa total $n\mu$); $\mu_0=0$.

En notación matemática, las transiciones adyacentes son:
$$
n \xrightarrow{\lambda} n+1\quad (0\le n\le k-1),\qquad\nn \xrightarrow{n\mu} n-1\quad (1\le n\le k).
$$


### Ecuaciones de balance (resumen)
- Estado n=0: $$\mu P_1 = \lambda P_0.$$
- Estado 0<n<k: $$\lambda P_{n-1} + (n+1)\mu P_{n+1} = (\lambda + n\mu) P_n.$$
- Estado n=k: $$\lambda P_{k-1} = k\mu P_k.$$


In [ ]:
# Preparación del entorno en el notebook (ajusta rutas si es necesario)
import sys
from pathlib import Path
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Repo root añadido a sys.path:', repo_root)


In [ ]:
# Imports teóricos y utilidades
import numpy as np
import matplotlib.pyplot as plt
# Importar las funciones del paquete (si están en src/)
try:
    from src.core.simulation import simulate_mmk_k, simulate_from_rates
    from src.core.theory import erlang_b, mmkk_stationary_probs, mmkk_mean_wait
    print('Import desde src.core OK')
except Exception as e:
    print('No fue posible importar src.core directamente:', e)


In [ ]:
# Algoritmo recursivo estable para Erlang B (B(k,A))
def erlang_b_recursive(A: float, k: int) -> float:
    B = 1.0
    for i in range(1, k+1):
        B = (A * B) / (i + A * B)
    return B

# Ejemplo de uso
for k_test in (1,2,5,10):
    for A_test in (0.5,1.0,2.0,10.0):
        print(f'k={k_test}, A={A_test}, B={erlang_b_recursive(A_test,k_test):.8e}')


### Validación: Teoría vs Simulación (gráfica)
La siguiente figura muestra la comparación entre la fórmula de Erlang B y los puntos empíricos obtenidos por simulación.

![Erlang B validation](plots/erlangb_validation_k10.png)


## Métricas complementarias y Ley de Little

A continuación calculamos las métricas relevantes para el sistema M/M/k/k (Erlang B):

- Tasa efectiva de llegadas: $\lambda_{\mathrm{eff}} = \lambda\, (1 - P_k)$ — tasa de transacciones que efectivamente entran al sistema.
- Número medio de clientes en el sistema: $L = \sum_{n=0}^k n\,P_n$.
- Tiempo medio en el sistema: $W = 1/\mu$ (ya que $W_q=0$).

Verificaremos numéricamente la Ley de Little: $L = \lambda_{\mathrm{eff}}\cdot W$.


In [ ]:
# Cálculo numérico de métricas para M/M/k/k
from src.core.theory import mmkk_stationary_probs, mmkk_mean_wait

# Parámetros de ejemplo (ajustables)
lam = 8.0
mu = 1.0
k = 10
A = lam / mu

Pn = mmkk_stationary_probs(lam, mu, k)
Pk = Pn[-1]
lambda_eff = lam * (1.0 - Pk)
L = sum(n * Pn[n] for n in range(len(Pn)))
W = 1.0 / mu  # porque W_q = 0

print(f"Parametros: lam={lam}, mu={mu}, k={k}, A={A}")
print("P_n (n=0..k):", [round(p, 6) for p in Pn])
print(f"P_k (probabilidad de bloqueo) = {Pk:.6f}")
print(f"Tasa efectiva lambda_eff = {lambda_eff:.6f}")
print(f"Numero medio en sistema L = {L:.6f}")
print(f"Tiempo medio en sistema W = {W:.6f}")
print("Verificación Ley de Little: L vs lambda_eff * W =>", L, lambda_eff * W)
print("Diferencia absoluta:", abs(L - lambda_eff * W))
print("Error relativo:", abs(L - lambda_eff * W) / (L if L>0 else 1.0))


## Conclusión: Aplicación a Pasarelas de Pago

En el contexto de una pasarela de pagos bancarios, cada "cliente" en el modelo representa una petición de transacción entrante, y cada uno de los $k$ servidores modela una instancia de servicio (microservicio) capaz de procesar transacciones en paralelo. La probabilidad de bloqueo $P_k$ cuantifica la fracción de transacciones rechazadas inmediatamente por falta de capacidad — es decir, cuando las $k$ instancias están ocupadas.

La tasa de tráfico ofrecido $A=\lambda/\mu$ es crítica: al incrementar $A$ (más solicitudes por unidad de tiempo, o menor capacidad de procesamiento por servidor), $P_k$ crece rápidamente y la tasa efectiva de aceptación $\lambda_{\mathrm{eff}}=\lambda(1-P_k)$ disminuye. Monitorear $A$ y $P_k$ permite dimensionar la infraestructura (escalar horizontalmente el número de servidores $k$ o mejorar la tasa de servicio $\mu$) para mantener el nivel de rechazo por debajo de un umbral aceptable. En producción esto se traduce en alertas APM y reglas de escalado automático basadas en métricas de bloqueo y tráfico.
